# Video Info

In [ ]:
import supervision as sv
VIDEO_PATH = ""
sv.VideoInfo.from_video_path(video_path = VIDEO_PATH)
frame_generator = sv.get_video_frames_generator(source_path=VIDEO_PATH) # stride = 2 to take only every second frame
frame = next(iter(frame_generator))
sv.plot_image(image=frame, size=(8, 8)) # To plot 

In [ ]:
START = sv.Point(0, 1500)
END = sv.Point(3840, 1500)

line_zone = sv.LineZone(start=START, end=END)
line_zone_annotator = sv.LineZoneAnnotator(thickness=4,text_thickness=4,text_scale=2)
annotated_frame = line_zone_annotator.annotate(annotated_frame, line_counter=line_zone)
sv.plot_image(annotated_frame, (12, 12))

In [ ]:
import supervision as sv
from supervision.metrics import F1Score

predictions = sv.Detections(...)
targets = sv.Detections(...)

f1_metric = F1Score()
f1_result = f1_metric.update(predictions, targets).compute()

print(f1_result)
print(f1_result.f1_50)
print(f1_result.small_objects.f1_50)

In [ ]:
# To save frames using sink
with sv.VideoSink(target_path=RESULT_VIDEO_PATH, video_info=video_info) as sink:
    for frame in sv.get_video_frames_generator(source_path=VIDEO_PATH, stride=2):
        sink.write_frame(frame=frame)


In [ ]:
# Detection dataset
ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/train/images",
    annotations_directory_path=f"{dataset.location}/train/labels",
    data_yaml_path=f"{dataset.location}/data.yaml",
)

In [ ]:
# Filter detections
detections_index = detections[0]
detections_index_list = detections[[0, 1, 3]]
detections_index_slice = detections[:2]
sv.Detections.from_inference(results).with_nms(threshold=0.1)

detections_filtered = detections[(detections.class_id != 0) & (detections.confidence > 0.7)]

detections.data["class_name"] = np.array(["person" for _ in range(len(detections))])
width, height = video_info.resolution_wh
frame_area = width * height
detections.area = frame_area

# For loop with detections
for class_id, confidence in zip(detections_filtered.class_id, detections_filtered.confidence):
    labels.append(f"{model.model.names[class_id]} {confidence:.2f}")


In [ ]:
# Visualise annotations
IMAGE_NAME = list(ds.images.keys())[0]

image = ds.images[IMAGE_NAME]
annotations = ds.annotations[IMAGE_NAME]

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
mask_annotator = sv.MaskAnnotator()

labels = [f"{ds.classes[class_id]}" for class_id in annotations.class_id]

annotated_image = mask_annotator.annotate(image.copy(), detections=annotations)
annotated_image = box_annotator.annotate(annotated_image, detections=annotations)
annotated_image = label_annotator.annotate(
    annotated_image, detections=annotations, labels=labels
)

sv.plot_image(image=annotated_image, size=(8, 8))

In [ ]:
byte_track = sv.ByteTrack(minimum_consecutive_frames=3)
byte_track.reset()
detections = byte_track.update_with_detections(detections)
smoother = sv.DetectionsSmoother()
detections = tracker.update_with_detections(detections)
detections = smoother.update_with_detections(detections)
trace_annotator = sv.TraceAnnotator()

In [ ]:
# To save detections to json file: empty detections will get skipped
json_sink = sv.JSONSink(FILE_NAME)
json_sink.open()
 

In [ ]:
pipeline = InferencePipeline.init(
    model_id=INFERENCE_MODEL,
    video_reference=SOURCE_VIDEO_PATH,
    on_prediction=callback,
    iou_threshold=IOU_THRESHOLD,
    confidence=CONFIDENCE_THRESHOLD,
)
pipeline.start()
pipeline.join()

In [ ]:
import supervision as sv
from inference.models.utils import get_roboflow_model

model = get_roboflow_model('yolov8n-640')
video_info = sv.VideoInfo.from_video_path(path_to_video)
label = sv.LabelAnnotator()
byte_tracker = sv.ByteTrack(frame_rate=video_info.fps)
frame_generator = sv.get_video_frames_generator(path_to_video)
frame = next(frame_generator)
result = model.infer(frame)[0]
detections = sv.Detections.from_inference(result)
tracked_detections = byte_tracker.update_with_detections(detections)
labels = [ f"{tracker_id}" for tracker_id in tracked_detections.tracker_id ]
annotated_frame = label.annotate(scene=frame.copy(), detections=tracked_detections, labels=labels)
sv.plot_image(annotated_frame)

In [ ]:
# convert json data to sv detection
def json_to_detections(json_file: str) -> List[sv.Detections]:
    rows_by_frame_number = defaultdict(list)
    with open(json_file, "r") as f:
        data = json.load(f)
    for row in data:
        frame_number = int(row["frame_number"])
        rows_by_frame_number[frame_number].append(row)

    detections_list = []
    for frame_number, rows in rows_by_frame_number.items():
        xyxy = []
        class_id = []
        confidence = []
        tracker_id = []
        custom_data = defaultdict(list)

        for row in rows:
            xyxy.append([row[key] for key in ["x_min", "y_min", "x_max", "y_max"]])
            class_id.append(row["class_id"])
            confidence.append(row["confidence"])
            tracker_id.append(row["tracker_id"])

            for custom_key in row.keys():
                if custom_key in ["x_min", "y_min", "x_max", "y_max", "class_id", "confidence", "tracker_id"]:
                    continue
                custom_data[custom_key].append(row[custom_key])

        if all([val == "" for val in class_id]):
            class_id = None
        if all([val == "" for val in confidence]):
            confidence = None
        if all([val == "" for val in tracker_id]):
            tracker_id = None

        detections_list.append(
            sv.Detections(
                xyxy=np.array(xyxy, dtype=np.float32),
                class_id=np.array(class_id, dtype=int),
                confidence=np.array(confidence, dtype=np.float32),
                tracker_id=np.array(tracker_id, dtype=int),
                data=dict(custom_data)
            )
        )
    
    return detections_list